#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
CATENETS_DIR = ROOT / "experiments" / "supplementary" / "catenets_custom"
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# install dependencies - only once
%pip install wandb loguru jax ott-jax gdown
%pip install -e "{CATENETS_DIR}"

In [ ]:
# custom library imports - restart kernel after installation if needed
from catenets.models.torch import FlexTENet

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def init_flextenet(input_dim, params, seed):
    return FlexTENet(
        # standard init
        n_unit_in=input_dim,
        binary_y=False,
        seed=seed,

        # model capacity
        n_layers_r=2,
        n_layers_out=2,
        n_units_s_r=params["hidden_dim"],
        n_units_p_r=params["hidden_dim"],
        n_units_s_out=params["hidden_dim"],
        n_units_p_out=params["hidden_dim"],

        private_out=False,
        shared_repr=False,

        # tuned parameters
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
        batch_size=params["batch_size"],

        # early stopping
        early_stopping=True,
        patience=params["patience"],
        n_iter=params["max_epochs"],
        val_split_prop=0.2,

        # model-specific settings
        penalty_orthogonal=0.01,
        clipping_value=1,
        dropout=False)

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/flextenet.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"size_{train_size}" / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data - .fit() uses internal train val split so merge
    _, _, train_df, val_df, _ = make_splits(df=df, train_size=train_size, seed=seed)
    train_df = pd.concat([train_df, val_df])

    # extract arrays
    X = train_df[confounders].to_numpy(dtype=np.float32)
    t = train_df["T"].to_numpy(dtype=np.float32)   
    y = train_df["Y"].to_numpy(dtype=np.float32) 

    # init model
    model = init_flextenet(input_dim, params, seed)

    # train
    model = model.fit(X, y, t)
    
    # checkpoint
    torch.save(model.state_dict(), ckpt_dir / "FlexNet.pt")

#### evaluation

In [ ]:
def load_flextenet(train_size, params, seed, confounders, device):
    input_dim = len(confounders)

    # set checkpoint path
    ckpt_path = ROOT / "experiments" / "supplementary" / "baselines" / "catenets_baselines" / "chkpts" / f"size_{train_size}" / f"seed_{seed}" / "FlexNet.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    # load model checkpoint
    model = init_flextenet(input_dim=input_dim, params=params, seed=seed,)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    model.eval()
    
    return model

In [ ]:
def get_estimates_flextenet(model, confounders, test_df):
    X = test_df[confounders].to_numpy(dtype=np.float32)
    tau_hat = model.predict(X).detach().cpu().numpy()
    test_df["flextenet_hat"] = tau_hat
    return test_df

In [ ]:
def compute_metrics_flextenet(eval_df):
    required = {"flextenet_hat", "cate"}

    # check if estimates are available
    missing = required - set(eval_df.columns)
    if missing:
        raise ValueError(f"eval_df is missing required columns: {sorted(missing)}")

    specs = [("FlexTENet", "flextenet_hat")]
    rows = []
    for name, score_col in specs:
        ranked = eval_df.sort_values(score_col, ascending=False).copy()

        rows.append({
            "model": name,
            "autoc": autoc(ranked),
            "policy_value": policy_value(ranked)})

    return pd.DataFrame(rows)

In [ ]:
# init collector
all_metrics = []

# loop over seeds and sizes
for seed in range(5):
    for size in [100, 250, 500, 1000, 2000]:

        # get testing data
        _, _, _, _, test_df = make_splits(df=df, train_size=size, seed=seed)

        # load model
        flextenet = load_flextenet(size, params, seed, confounders, device)
        df_eval = get_estimates_flextenet(flextenet, confounders, test_df)
        df_metrics = compute_metrics_flextenet(df_eval)

        # store
        df_metrics["size"] = size
        df_metrics["seed"] = seed
        all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
summary = (df_all.groupby(["size", 'model']).agg(['mean', 'std']).reset_index())